In [7]:
# ================================================
# 1. INSTALL REQUIRED LIBRARIES
# ================================================

%env GROQ_API_KEY=your_real_key_here

!pip install requests beautifulsoup4 ipywidgets --quiet

import requests
from bs4 import BeautifulSoup
import re
import ipywidgets as widgets
from IPython.display import display
import os
import time

# ================================================
# 2. GROQ API KEY (USE YOUR OWN)
# ================================================
import os
api_key = os.environ.get("GROQ_API_KEY")  # safely gets the key

os.environ["GROQ_API_KEY"] = API_KEY
API_URL = "https://api.groq.com/openai/v1/chat/completions"

# ================================================
# 3. TEXT CLEANING FUNCTION
# ================================================
def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^A-Za-z0-9 .,-]", "", text)
    return text.strip()

# ================================================
# 4. WEB SCRAPING FUNCTION
# ================================================
def scrape_web(query):
    sources = [
        f"https://www.bbc.co.uk/search?q={query}",
        f"https://www.reuters.com/site-search/?query={query}"
    ]
    combined_text = ""
    for url in sources:
        try:
            res = requests.get(url, timeout=6)
            soup = BeautifulSoup(res.text, "html.parser")
            paragraphs = soup.find_all("p")
            extracted = " ".join([p.get_text() for p in paragraphs[:5]])
            combined_text += extracted + " "
        except Exception:
            pass
        time.sleep(0.5)
    return combined_text.strip()

# ================================================
# 5. CALL LLM USING GROQ API
# ================================================
def verify_with_llm(news_text, scraped_text):
    prompt = f"""
Compare the following news with trusted scraped information.

NEWS PROVIDED BY USER:
{news_text}

INFORMATION FROM RELIABLE SOURCES:
{scraped_text}

Give the result in this format:
Authenticity Score: (0-100)
Final Decision: REAL / FAKE / POSSIBLY MISLEADING
Explanation: (2–3 lines)
"""
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    data = {
        "model": "llama-3.3-70b-versatile",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.2,
        "max_tokens": 400
    }
    try:
        response = requests.post(API_URL, headers=headers, json=data, timeout=30)
        if response.status_code != 200:
            return f"API Request Failed: {response.status_code} - {response.text}"
        resp = response.json()
        return resp["choices"][0]["message"]["content"].strip()
    except Exception as e:
        return f"API Error: {e}"

# ================================================
# 6. DASHBOARD UI (SMALL DIFFERENCE)
# ================================================
title_html = widgets.HTML(
    value="""
    <h1 style='color:#1b1464;'>🔎 AI News Verifier</h1>
    <h3 style='color:#6a89cc;'>Check your news authenticity</h3>
    <hr style='border:1px solid #d1d8e0;'>
    """
)

input_box = widgets.Textarea(
    placeholder="Paste your news here...",
    description="📰 News:",
    layout=widgets.Layout(width="100%", height="140px"),
    style={'description_width': 'initial'}
)

run_button = widgets.Button(
    description="Check News",
    button_style="info",
    icon="search",
    layout=widgets.Layout(width="180px")
)

output_area = widgets.HTML(
    value="",
    layout=widgets.Layout(width="100%", padding="12px", border="1px solid #d1d8e0", border_radius="8px", background_color="#f1f2f6")
)

# ================================================
# 7. BUTTON LOGIC
# ================================================
def on_button_click(b):
    news = input_box.value.strip()
    if len(news) < 10:
        output_area.value = "<b style='color:red;'>Please enter a valid news text.</b>"
        return

    output_area.value = "<b>🔍 Scraping trusted sources...</b>"
    scraped = scrape_web(news[:50])

    output_area.value = "<b>🤖 Analyzing with AI...</b>"
    cleaned = clean_text(news)
    result = verify_with_llm(cleaned, scraped)

    output_area.value = f"""
    <h3 style='color:#27ae60;'>✅ RESULT</h3>
    <div style='padding:15px; background:#f0f0f5; border-radius:10px; border-left:4px solid #1b1464; color:#2d3436'>
        {result.replace('\n', '<br>')}
    </div>
    """

run_button.on_click(on_button_click)

# ================================================
# 8. DISPLAY UI
# ================================================
display(widgets.VBox([title_html, input_box, run_button, output_area]))


env: GROQ_API_KEY=your_real_key_here
